In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Lang Smith 토큰 발급하기

In [2]:
# 설치
pip install -U langchain langchain_openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.4/120.4 kB 3.9 MB/s eta 0:00:00


In [4]:
%cd /content/drive/MyDrive/Colab Notebooks/인사교1_LangChain_20260624

/content/drive/MyDrive/Colab Notebooks/인사교1_LangChain_20260624


In [3]:
import os

In [5]:
with open("./key/lang_smith_key.txt", "r") as f:
    api_key = f.read().strip()

LANGSMITH_TRACING="true"
LANGSMITH_ENDPOINT="https://api.smith.langchain.com"
LANGSMITH_API_KEY=api_key
LANGSMITH_PROJECT="aischool0701"

os.environ['LANGSMITH_TRACING'] = LANGSMITH_TRACING
os.environ['LANGSMITH_ENDPOINT'] = LANGSMITH_ENDPOINT
os.environ['LANGSMITH_API_KEY'] = LANGSMITH_API_KEY
os.environ['LANGSMITH_PROJECT'] = LANGSMITH_PROJECT

In [6]:
with open("./key/openai_key.txt", "r") as f:
    api_key = f.read().strip()

os.environ['OPENAI_API_KEY'] = api_key

In [7]:
# 기본 체인 만들어서 호출하기
from langchain.chat_models import init_chat_model
# 프롬프트
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
# 아웃풋 파서
from langchain_core.output_parsers import StrOutputParser

In [14]:
# 모델생성
llm_4o_mini = init_chat_model("openai:gpt-4o-mini", max_tokens=1024)
# 프롬프트 템플릿 생성
template = ChatPromptTemplate.from_messages([
    ("system", "아래 사용자 입력 글을 스페인어로 번역해줘. 출력형식은 다른 멘트없이 번역글만 나오게해줘"),
    ("human", "userInput : {input}")
])
# 문자열파서 생성
strParser = StrOutputParser()

In [15]:
# 체인 구성하기
translate_chain = template | llm_4o_mini | strParser

In [16]:
# 체인 호출하기
print(translate_chain.invoke({"input": "노란 티셔츠를 입은 아이는 너의 사촌이야?"}))

¿El niño que lleva una camiseta amarilla es tu primo?


### Lang Serve 활용하기

In [17]:
!pip install langserve fastapi uvicorn pyngrok

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 14.7 MB/s eta 0:00:00


In [20]:
!pip install sse_starlette

#### FastAPI + LangServe 구성하기

In [18]:
from fastapi import FastAPI
from langserve import add_routes

In [24]:
# 서버생성
app = FastAPI()

add_routes(
    app,                # 라우팅을 추가할 서버
    translate_chain,    # 라우팅에 연결할 체인
    path="/query"       # 연결된 체인을 호출할때 활용하는 URL
)

#### Colab에서 서버 실행

In [25]:
import nest_asyncio             # 노트북 커널과 별개 수행이 가능하도록 도와주는 패키지
import uvicorn                  # 서버를 구동하는 패키지
from threading import Thread    # 기존 노트북 커널과 별도로 쓰레드가 분할되도록 하는 도구

In [26]:
nest_asyncio.apply()

def run() :
    uvicorn.run(
        app,            # 구동할 서버 앱
        host="0.0.0.0", # 현재 할당받은 컴퓨팅 지원의 IP를 지칭
        port=8000       # 사용할 포트번호
    )

thread = Thread(target=run) # 쓰레드 생성
thread.start()              # 구동

#### colab 인스턴스 내부에서 호출하기

In [27]:
!curl -X POST "http://0.0.0.0:8000/query/invoke" \
-H "Content-Type: application/json" \
-d '{"input" : {"input": "노란 티셔츠를 입은 아이는 너의 사촌이야?"}}'

INFO:     127.0.0.1:52282 - "POST /query/invoke HTTP/1.1" 200 OK
{"output":"¿El niño que lleva una camiseta amarilla es tu primo?","metadata":{"run_id":"ae1106fd-e19f-43d8-91b4-e3c0dde5e9d1","feedback_tokens":[]}}

#### ngrok을 활용한 외부에서 호출

In [43]:
import os
with open("./key/ngrok_key.txt", "r") as f:
   api_key = f.read().strip()
os.system(f'ngrok config add-authtoken {api_key}')

0

In [44]:
from pyngrok import ngrok

In [45]:
public_url = ngrok.connect(8000)

In [46]:
public_url

<NgrokTunnel: "https://federal-onto-speckled.ngrok-free.dev" -> "http://localhost:8000">

In [42]:
# 연결해제
ngrok.disconnect(public_url)

In [36]:
!curl -X POST "https://federal-onto-speckled.ngrok-free.dev:8000/query/invoke" \
-H "Content-Type: application/json" \
-d '{"input" : {"input": "노란 티셔츠를 입은 아이는 너의 사촌이야?"}}'

^C
